# Train And Evaluate Models On Colab

Run one model selector cell (`LSTM`, `LSTM2`, or `TCT-GAT`), then run the shared setup, dataset copy, verification, W&B login, train, and evaluate cells. The train command keeps the same `!python -m {TRAIN_MODULE}` format as the LSTM notebook.

## 1. Mount Drive

In [1]:
%cd /content
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Select Model

Run exactly one selector cell before continuing.

### LSTM

In [ ]:
MODEL_KIND = "lstm"
DATASET_VERSION = "tts_lstm"       # examples: lstm_v1, lstm_v2, tts_lstm
MODEL_VERSION = "tts_lstm"         # controls config/checkpoint/model/log paths
BASE_DATASET_NAME = "base"
PROCESSED_DIR_NAME = "lstm_processed"
ARTIFACT_FAMILY = "lstm"
CONFIG_FAMILY = "lstm"
TRAIN_MODULE = "src.training.lstm_training.train_lstm"
EVAL_MODULE = "src.training.lstm_training.evaluate"
DATASET_MODULE = None
BATCH_SIZE = 32768
EVAL_BATCH_SIZE = 32768
CHECKPOINT_NAME = "best.pt"
EVAL_SPLIT = "test_2024_april_june"  # val, test_2025_winter, test_2024_april_june
MAX_BATCHES = None

### LSTM2

In [ ]:
MODEL_KIND = "lstm2"
DATASET_VERSION = "tts_lstm2"
MODEL_VERSION = "tts_lstm2"
BASE_DATASET_NAME = "base"
PROCESSED_DIR_NAME = "lstm2_processed"
ARTIFACT_FAMILY = "lstm2"
CONFIG_FAMILY = "lstm2"
TRAIN_MODULE = "src.training.lstm2_training.train_lstm2"
EVAL_MODULE = "src.training.lstm2_training.evaluate_lstm2"
DATASET_MODULE = "src.data.lstm2.lstm2_dataset"
BATCH_SIZE = 32768
EVAL_BATCH_SIZE = 8192
CHECKPOINT_NAME = "best.pt"
EVAL_SPLIT = "test_2025_winter"  # val, test_2025_winter, test_2024_april_june
MAX_BATCHES = None
AUTOREGRESSIVE_ROLLOUT = False
ROLLOUT_HORIZONS = 8
MONTE_CARLO_SAMPLES = 1
WEATHER_NOISE_EVAL = False

### TCT-GAT

In [2]:
MODEL_KIND = "tct_gat"
DATASET_VERSION = "tct_gat1_ar"
MODEL_VERSION = "tct_gat1_ar"
BASE_DATASET_NAME = None
PROCESSED_DIR_NAME = "tct_gat_processed"
ARTIFACT_FAMILY = "tct_gat"
CONFIG_FAMILY = "tct_gat"
TRAIN_MODULE = "src.training.tct_gat_training.train_tct_gat"
EVAL_MODULE = "src.training.tct_gat_training.evaluate_tct_gat"
DATASET_MODULE = "src.data.tct_gat.tct_gat_dataset"
BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
CHECKPOINT_NAME = "best.pt"
EVAL_SPLIT = "test_2025_winter"  # val, test_2025_winter, test_2024_april_june
MAX_BATCHES = None
AUTOREGRESSIVE_ROLLOUT = False
ROLLOUT_HORIZONS = 8
MONTE_CARLO_SAMPLES = 1
WEATHER_NOISE_EVAL = False

## 3. Prepare Code And Paths

In [3]:
from pathlib import Path
import os
import shutil
import subprocess

# Code should live on local Colab disk. Drive is only for source data/checkpoints/logs.
GITHUB_REPO_URL = "https://github.com/sungjelly/Seoul_bike_project.git"
GITHUB_BRANCH = "main"
CODE_DIR = Path("/content/Seoul_bike_project")
RELOAD_LOCAL_CODE = True
RELOAD_LOCAL_DATA = True

DRIVE_ROOT = Path("/content/drive/MyDrive/Seoul_bike_project")

if CODE_DIR.exists() and RELOAD_LOCAL_CODE:
    shutil.rmtree(CODE_DIR)

if not CODE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(CODE_DIR)],
        check=True,
    )

PROJECT_DIR = CODE_DIR
CONFIG_PATH = f"configs/models/{CONFIG_FAMILY}/{MODEL_VERSION}.yaml"

DRIVE_PROCESSED_ROOT = DRIVE_ROOT / "data" / PROCESSED_DIR_NAME
LOCAL_PROCESSED_ROOT = PROJECT_DIR / "data" / PROCESSED_DIR_NAME
DRIVE_DATA_DIR = DRIVE_PROCESSED_ROOT / DATASET_VERSION
LOCAL_DATA_DIR = LOCAL_PROCESSED_ROOT / DATASET_VERSION
DATA_DIR = LOCAL_DATA_DIR

DRIVE_BASE_DATA_DIR = DRIVE_PROCESSED_ROOT / BASE_DATASET_NAME if BASE_DATASET_NAME else None
LOCAL_BASE_DATA_DIR = LOCAL_PROCESSED_ROOT / BASE_DATASET_NAME if BASE_DATASET_NAME else None
GRAPH_DIR = DATA_DIR / "graph"

CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / ARTIFACT_FAMILY / MODEL_VERSION
MODEL_DIR = DRIVE_ROOT / "models" / ARTIFACT_FAMILY / MODEL_VERSION
LOG_DIR = DRIVE_ROOT / "logs" / ARTIFACT_FAMILY / MODEL_VERSION
CHECKPOINT_PATH = CHECKPOINT_DIR / CHECKPOINT_NAME

os.chdir(PROJECT_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Model kind:", MODEL_KIND)
print("Code directory:", PROJECT_DIR)
print("Drive artifact root:", DRIVE_ROOT)
print("Drive dataset:", DRIVE_DATA_DIR)
if DRIVE_BASE_DATA_DIR is not None:
    print("Drive base dataset:", DRIVE_BASE_DATA_DIR)
print("Local dataset:", DATA_DIR)
if MODEL_KIND == "tct_gat":
    print("Graph directory:", GRAPH_DIR)
print("Training module:", TRAIN_MODULE)
print("Evaluation module:", EVAL_MODULE)
print("Config path:", CONFIG_PATH)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Checkpoint path:", CHECKPOINT_PATH)
print("Current working directory:", Path.cwd())

Model kind: tct_gat
Code directory: /content/Seoul_bike_project
Drive artifact root: /content/drive/MyDrive/Seoul_bike_project
Drive dataset: /content/drive/MyDrive/Seoul_bike_project/data/tct_gat_processed/tct_gat1_ar
Local dataset: /content/Seoul_bike_project/data/tct_gat_processed/tct_gat1_ar
Graph directory: /content/Seoul_bike_project/data/tct_gat_processed/tct_gat1_ar/graph
Training module: src.training.tct_gat_training.train_tct_gat
Evaluation module: src.training.tct_gat_training.evaluate_tct_gat
Config path: configs/models/tct_gat/tct_gat1_ar.yaml
Checkpoint directory: /content/drive/MyDrive/Seoul_bike_project/checkpoints/tct_gat/tct_gat1_ar
Checkpoint path: /content/drive/MyDrive/Seoul_bike_project/checkpoints/tct_gat/tct_gat1_ar/best.pt
Current working directory: /content/Seoul_bike_project


## 4. Install Requirements

In [4]:
%cd /content/Seoul_bike_project
!python -m pip install -r requirements.txt

/content


## 5. Copy Dataset To Colab Disk

In [5]:
import shutil

copy_pairs = []
if DRIVE_BASE_DATA_DIR is not None:
    copy_pairs.append((DRIVE_BASE_DATA_DIR, LOCAL_BASE_DATA_DIR))
copy_pairs.append((DRIVE_DATA_DIR, LOCAL_DATA_DIR))

missing = [str(src) for src, _ in copy_pairs if not src.exists()]
if missing:
    raise FileNotFoundError("Missing Drive dataset directories:\n" + "\n".join(missing))

if MODEL_KIND == "tct_gat":
    required_graph_files = [
        DRIVE_DATA_DIR / "graph" / "neighbor_index_rr.npy",
        DRIVE_DATA_DIR / "graph" / "edge_attr_rr.npy",
    ]
    missing_graph = [str(path) for path in required_graph_files if not path.exists()]
    if missing_graph:
        raise FileNotFoundError("Missing TCT-GAT graph files:\n" + "\n".join(missing_graph))

LOCAL_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
for src, dst in copy_pairs:
    if dst.exists() and RELOAD_LOCAL_DATA:
        shutil.rmtree(dst)
    if not dst.exists():
        shutil.copytree(src, dst)
        print(f"Copied {src} -> {dst}")
    else:
        print(f"Using existing local copy: {dst}")

print("Using local dataset:", DATA_DIR)

Copied /content/drive/MyDrive/Seoul_bike_project/data/tct_gat_processed/tct_gat1_ar -> /content/Seoul_bike_project/data/tct_gat_processed/tct_gat1_ar
Using local dataset: /content/Seoul_bike_project/data/tct_gat_processed/tct_gat1_ar


## 6. Verify Dataset

In [6]:
import json
import numpy as np

if MODEL_KIND == "lstm":
    base_data_path = DATA_DIR / "base_data.json"
    array_dir = DATA_DIR
    if base_data_path.exists():
        base_data = json.loads(base_data_path.read_text())
        array_dir = (DATA_DIR / str(base_data["base_data_dir"]).replace("\\", "/")).resolve()

    required = [
        array_dir / "dynamic_features.npy",
        array_dir / "targets.npy",
        array_dir / "targets_raw.npy",
        array_dir / "static_numeric.npy",
        DATA_DIR / "sample_index_train.npy",
        DATA_DIR / "sample_index_val.npy",
        DATA_DIR / "feature_config.json",
        DATA_DIR / "dataset_summary.json",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing LSTM dataset files:\n" + "\n".join(missing))
    summary = json.loads((DATA_DIR / "dataset_summary.json").read_text())
    print(json.dumps({
        "T_total": summary["T_total"],
        "S": summary["S"],
        "samples_per_split": summary["samples_per_split"],
    }, indent=2))

elif MODEL_KIND == "lstm2":
    array_dir = LOCAL_BASE_DATA_DIR
    dynamic = np.load(array_dir / "dynamic_features.npy", mmap_mode="r")
    target_time = np.load(array_dir / "target_time_features.npy", mmap_mode="r")
    targets = np.load(array_dir / "targets.npy", mmap_mode="r")
    targets_raw = np.load(array_dir / "targets_raw.npy", mmap_mode="r")
    feature_config = json.loads((LOCAL_DATA_DIR / "feature_config.json").read_text())

    assert dynamic.ndim == 3 and dynamic.shape[-1] == 7, dynamic.shape
    assert target_time.ndim == 2 and target_time.shape[-1] == 8, target_time.shape
    assert targets.ndim == 4 and targets.shape[-2:] == (8, 2), targets.shape
    assert targets_raw.ndim == 4 and targets_raw.shape[-2:] == (8, 2), targets_raw.shape
    assert len(feature_config["dynamic_feature_columns"]) == 7
    assert len(feature_config["target_time_feature_columns"]) == 8
    assert (LOCAL_DATA_DIR / "sample_index_train.npy").exists()
    assert (LOCAL_DATA_DIR / "sample_index_val.npy").exists()

    print("dynamic_features:", dynamic.shape)
    print("target_time_features:", target_time.shape)
    print("targets:", targets.shape)
    print("targets_raw:", targets_raw.shape)
    print("dynamic columns:", feature_config["dynamic_feature_columns"])
    print("target time columns:", feature_config["target_time_feature_columns"])

elif MODEL_KIND == "tct_gat":
    summary = json.loads((DATA_DIR / "dataset_summary.json").read_text())
    feature_config = json.loads((DATA_DIR / "feature_config.json").read_text())
    graph_summary = json.loads((GRAPH_DIR / "graph_summary.json").read_text())
    rental = np.load(DATA_DIR / "rental_features.npy", mmap_mode="r")
    returns = np.load(DATA_DIR / "return_features.npy", mmap_mode="r")
    targets = np.load(DATA_DIR / "targets.npy", mmap_mode="r")
    targets_raw = np.load(DATA_DIR / "targets_raw.npy", mmap_mode="r")
    future_weather = np.load(DATA_DIR / "future_weather_features.npy", mmap_mode="r")
    target_time = np.load(DATA_DIR / "target_time_features.npy", mmap_mode="r")
    neighbor_rr = np.load(GRAPH_DIR / "neighbor_index_rr.npy", mmap_mode="r")
    edge_rr = np.load(GRAPH_DIR / "edge_attr_rr.npy", mmap_mode="r")

    assert rental.ndim == 3 and rental.shape[-1] == 5, rental.shape
    assert returns.shape == rental.shape, returns.shape
    assert targets.ndim == 3 and targets.shape[-1] == 2, targets.shape
    assert targets_raw.shape == targets.shape, targets_raw.shape
    assert future_weather.ndim == 2 and future_weather.shape[-1] == 4, future_weather.shape
    assert target_time.ndim == 2 and target_time.shape[-1] == 8, target_time.shape
    assert neighbor_rr.shape[0] == rental.shape[1], neighbor_rr.shape
    assert edge_rr.shape[:2] == neighbor_rr.shape, edge_rr.shape
    assert "net_demand" not in json.dumps(feature_config, ensure_ascii=False)
    assert (DATA_DIR / "sample_time_index_train.npy").exists()
    assert (DATA_DIR / "sample_time_index_val.npy").exists()

    print("rental_features:", rental.shape)
    print("return_features:", returns.shape)
    print("targets:", targets.shape)
    print("future_weather_features:", future_weather.shape)
    print("target_time_features:", target_time.shape)
    print("graph rr:", neighbor_rr.shape, edge_rr.shape)
    print("samples_per_split:", summary["samples_per_split"])
    print("k_neighbors:", graph_summary["k_neighbors"])
else:
    raise ValueError(f"Unknown MODEL_KIND: {MODEL_KIND}")

rental_features: (21888, 2780, 5)
return_features: (21888, 2780, 5)
targets: (21888, 2780, 2)
future_weather_features: (21888, 4)
target_time_features: (21888, 8)
graph rr: (2780, 32) (2780, 32, 27)
samples_per_split: {'train': 12096, 'val': 1488, 'test_2025_winter': 2928, 'test_2024_april_june': 3360}
k_neighbors: 32


## 7. Dataset Smoke Check

In [7]:
if DATASET_MODULE is not None:
    !python -m {DATASET_MODULE} \
      --data-dir {DATA_DIR} \
      --split train \
      --batch-size 1

rental_seq.shape=(1, 2780, 33, 5)
return_seq.shape=(1, 2780, 33, 5)
y.shape=(1, 2780, 2)
target_time_features.shape=(1, 8)
future_weather_features.shape=(1, 4)
static_numeric.shape=(2780, 3)
district_id.shape=(2780,)
operation_type_id.shape=(2780,)
neighbor_index_rr.shape=(2780, 32)
neighbor_index_dd.shape=(2780, 32)
neighbor_index_rd.shape=(2780, 32)
neighbor_index_dr.shape=(2780, 32)
edge_attr_rr.shape=(2780, 32, 27)
edge_attr_dd.shape=(2780, 32, 27)
edge_attr_rd.shape=(2780, 32, 27)
edge_attr_dr.shape=(2780, 32, 27)


## 8. W&B Login

In [8]:
import os
from getpass import getpass
import wandb

wandb_key = os.environ.get("WANDB_API_KEY")

try:
    from google.colab import userdata
    wandb_key = wandb_key or userdata.get("WANDB_API_KEY")
except Exception as exc:
    print("Colab Secrets not available from this kernel:", type(exc).__name__)

if not wandb_key:
    wandb_key = getpass("Paste W&B API key: ")

os.environ["WANDB_API_KEY"] = wandb_key.strip()
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

Colab Secrets not available from this kernel: TimeoutException


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sungjelly (sungjelly-kaist-digital-humanities-and-social-sciences-g) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 9. Train With Source Script

In [ ]:
!python -m {TRAIN_MODULE} \
  --config {CONFIG_PATH} \
  --data_dir {DATA_DIR} \
  --checkpoint_dir {CHECKPOINT_DIR} \
  --model_dir {MODEL_DIR} \
  --log_dir {LOG_DIR} \
  --batch_size 32 \
  --resume auto \
  --resume_mode auto \
  --wandb_enabled true \
  --smoke_test true

Smoke test passed. loss=0.415142
No checkpoint found in /content/drive/MyDrive/Seoul_bike_project/checkpoints/tct_gat/tct_gat1_ar; starting fresh.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: sungjelly (sungjelly-kaist-digital-humanities-and-social-sciences-g) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run mzj1pq7l (0.5s)
wandb: ⣯ setting up run mzj1pq7l (0.5s)
wandb: ⣟ setting up run mzj1pq7l (0.5s)
wandb: ⡿ setting up run mzj1pq7l (0.5s)
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /content/Seoul_bike_project/wandb/run-20260510_122330-mzj1pq7l
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run tct_gat1_ar
wandb: ⭐️ View project at https://wandb.ai/sungjelly-kaist-digit

## 10. Show Final Training Metrics

In [ ]:
import json

final_metrics_path = LOG_DIR / "final_metrics.json"
if final_metrics_path.exists():
    print(json.dumps(json.loads(final_metrics_path.read_text()), indent=2))
else:
    print("final_metrics.json not found:", final_metrics_path)

## 11. Evaluate With Source Script

In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Missing checkpoint: {CHECKPOINT_PATH}")

if MODEL_KIND == "tct_gat":
    assert (DATA_DIR / f"sample_time_index_{EVAL_SPLIT}.npy").exists(), EVAL_SPLIT
else:
    assert (DATA_DIR / f"sample_index_{EVAL_SPLIT}.npy").exists(), EVAL_SPLIT

In [ ]:
!python -m {EVAL_MODULE} \
  --config {CONFIG_PATH} \
  --data_dir {DATA_DIR} \
  --checkpoint_path {CHECKPOINT_PATH} \
  --log_dir {LOG_DIR} \
  --split {EVAL_SPLIT} \
  --batch_size {EVAL_BATCH_SIZE} \
  --wandb_enabled true

## 12. Future Weather Uncertainty Evaluation

In [ ]:
if MODEL_KIND == "lstm":
    print("Future-weather uncertainty rollout is not implemented for the base LSTM evaluator.")
elif MODEL_KIND == "lstm2" and MODEL_VERSION == "tts_lstm2":
    print("LSTM2 future-weather rollout is only supported by tts_lstm2_v2-style configs.")
else:
    print("Run the next cell for noisy future-weather autoregressive rollout evaluation.")

In [ ]:
if MODEL_KIND == "tct_gat" or (MODEL_KIND == "lstm2" and MODEL_VERSION != "tts_lstm2"):
    !python -m {EVAL_MODULE} \
      --config {CONFIG_PATH} \
      --data_dir {DATA_DIR} \
      --checkpoint_path {CHECKPOINT_PATH} \
      --log_dir {LOG_DIR} \
      --split {EVAL_SPLIT} \
      --batch_size {EVAL_BATCH_SIZE} \
      --wandb_enabled true \
      --autoregressive_rollout true \
      --rollout_horizons {ROLLOUT_HORIZONS} \
      --monte_carlo_samples {MONTE_CARLO_SAMPLES} \
      --weather_noise_eval true
else:
    print("Skipped noisy future-weather rollout for this model/config.")

## 13. Show Evaluation Metrics

In [ ]:
import json

eval_metrics_path = LOG_DIR / f"{EVAL_SPLIT}_metrics.json"
if eval_metrics_path.exists():
    print(json.dumps(json.loads(eval_metrics_path.read_text()), indent=2))
else:
    print("Evaluation metrics not found:", eval_metrics_path)